# Study 814 — Trailing-Sharpe Anomaly 📐📈

**Does risk-adjusting momentum — ranking on the trailing *Sharpe ratio* — beat plain
momentum, or is it the same trade in a nicer suit?**

Risk-adjusted momentum (Rachev / Biglova et al) replaces Jegadeesh-Titman's raw past-return
ranking with a **reward-to-risk** ranking: each name's **trailing 12-month Sharpe** (mean ÷
std of daily returns, skipping the most recent month). Long the high-Sharpe names, short the
low-Sharpe ones. We take the self-contained daily version on a liquid US cross-section
(2010-01-04 → 2026-06-30, 50 names) and ask the honest question head-on:
**does the division earn its keep?**

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea — and the catch

A Sharpe ratio is *momentum numerator ÷ volatility denominator*. Ranking on it is supposed to keep the **high-quality** winners and drop the jittery ones. But notice: if the winners and the high-Sharpe names are mostly the **same** names, you have not built a new signal — you have re-labelled momentum. So the test is not 'does the Sharpe book make money' but 'does it **beat plain momentum**?'

In [1]:
R = dict(spread_bps=1.29, t_nw=0.83, mom_bps=1.59, mom_t=0.99, rho_sharpe_mom=0.953,
         rho_sharpe_negvol=0.088, gross_sharpe=0.2)
print('long high-Sharpe / short low-Sharpe : %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('plain 12-1 momentum, same sort      : %+.2f bps/day (NW t = %+.2f)'
      % (R['mom_bps'], R['mom_t']))
print('rank corr  Sharpe ~ momentum        : %+.3f' % R['rho_sharpe_mom'])
print('rank corr  Sharpe ~ (-vol)          : %+.3f' % R['rho_sharpe_negvol'])

long high-Sharpe / short low-Sharpe : +1.29 bps/day (NW t = +0.83)
plain 12-1 momentum, same sort      : +1.59 bps/day (NW t = +0.99)
rank corr  Sharpe ~ momentum        : +0.953
rank corr  Sharpe ~ (-vol)          : +0.088


The Sharpe sort is **0.95 rank-correlated with plain momentum** and carries almost none of the low-vol tilt (+0.09). It *is* momentum — and it earns a touch **less** (+1.29 vs +1.59 bps/day). Risk-adjusting bought nothing.

## 2. Is the sort just lucky? A live synthetic control

We plant a high-Sharpe → high-return effect in a seeded toy world (`edge>0`, via a persistent volatility tilt) and check the detector recovers it — and stays *silent* on the null (`edge=0`, Sharpe varies but is unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from trailing_sharpe import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=814, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=814, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.27  (should be ~0)
planted world: spread NW t = +6.64  (should light up)


## 3. The honest verdict — risk-adjusting is not a free lunch

On this liquid mega-cap tape the long-high-Sharpe / short-low-Sharpe spread is **+1.29 bps/day** with NW *t* = **+0.83** — the right sign, but **not significant**. And it does not beat what it is built from: at **0.95** rank correlation with plain 12-1 momentum, the Sharpe sort simply re-picks the same winners, earning a hair *less* than momentum itself (+1.59 bps, *t* +0.99) — and neither clears |t| ≥ 2. The observed spread sits only ~1.2σ into a 1,000-permutation placebo. The synthetic control recovers a *planted* effect cleanly, so this is a real 'nothing here', not a bug. **Signal: None** (momentum repackaged), **Tradability: Mirage** (net-negative at 1 bp).